# 01 — Data Exploration

Load all three datasets, inspect structure, visualise distributions and the contextual patterns that define what 'normal' means in each domain.

In [ ]:
import sys
from pathlib import Path

# Ensure project root is on sys.path regardless of cwd
_ROOT = Path(__file__).resolve().parent.parent if "__file__" in dir() else Path.cwd().parent
if str(_ROOT) not in sys.path:
    sys.path.insert(0, str(_ROOT))

import json
import warnings

warnings.filterwarnings("ignore")

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

FIGURES = _ROOT / "outputs" / "figures"
OUTPUTS = _ROOT / "outputs"
FIGURES.mkdir(parents=True, exist_ok=True)
OUTPUTS.mkdir(parents=True, exist_ok=True)

from src.datasets import load_network, load_manufacturing, make_synthetic_credit_card
from src.detectors import (
    IsolationForestDetector,
    LOFDetector,
    OneClassSVMDetector,
    DBSCANDetector,
    MahalanobisDetector,
    AutoencoderDetector,
)
from src.evaluation import (
    run_arena,
    sweep_contamination,
    evaluate_detector,
    auprc,
    precision_recall_f1_at_contamination,
)
import src.visualisation as vis


## Load datasets

In [ ]:
# Use synthetic credit card so the notebook runs without downloading the CSV.
# Swap in load_credit_card() after running `make data`.
X_cc, y_cc = make_synthetic_credit_card(n_samples=5000, random_state=42)
X_net, y_net = load_network(n_samples=3000, random_state=42)
X_mfg, y_mfg = load_manufacturing(n_samples=2000, random_state=42)

for name, X, y in [("credit_card", X_cc, y_cc), ("network", X_net, y_net), ("manufacturing", X_mfg, y_mfg)]:
    print(f"{name:15s}  shape={X.shape}  anomalies={y.sum()} ({y.mean():.3%})")


## Dataset summary → `outputs/01_dataset_summary.json`

In [ ]:
summary = {}
for name, X, y in [("credit_card", X_cc, y_cc), ("network", X_net, y_net), ("manufacturing", X_mfg, y_mfg)]:
    X_n, X_a = X[y == 0], X[y == 1]
    summary[name] = {
        "n_samples": int(len(X)),
        "n_features": int(X.shape[1]),
        "n_anomalies": int(y.sum()),
        "anomaly_rate": float(y.mean()),
        "feature_means_normal": X_n.mean(axis=0).tolist(),
        "feature_stds_normal": X_n.std(axis=0).tolist(),
        "feature_means_anomaly": X_a.mean(axis=0).tolist(),
        "feature_stds_anomaly": X_a.std(axis=0).tolist(),
    }

with (OUTPUTS / "01_dataset_summary.json").open("w") as f:
    json.dump(summary, f, indent=2)
print("Saved outputs/01_dataset_summary.json")
pd.DataFrame(
    {n: {"n_samples": v["n_samples"], "n_features": v["n_features"],
         "n_anomalies": v["n_anomalies"], "anomaly_rate_pct": f"{v['anomaly_rate']:.3%}"}
     for n, v in summary.items()}
).T


## Feature distributions: normal vs anomaly

In [ ]:
fig, axes = plt.subplots(3, 5, figsize=(16, 9))
datasets_list = [
    ("Credit Card (synthetic)", X_cc, y_cc),
    ("Network Intrusion", X_net, y_net),
    ("Manufacturing Sensor", X_mfg, y_mfg),
]
feature_labels = {
    "Credit Card (synthetic)": [f"V{i}" for i in range(1, 6)],
    "Network Intrusion": ["hour_sin", "hour_cos", "req_rate", "sess_dur", "subnet"],
    "Manufacturing Sensor": ["raw_temp", "hour_sin", "hour_cos", "seasonal_exp", "residual"],
}
for row, (title, X, y) in enumerate(datasets_list):
    labels = feature_labels[title]
    for col in range(5):
        ax = axes[row, col]
        ax.hist(X[y == 0, col], bins=30, alpha=0.6, color="steelblue", density=True, label="normal")
        ax.hist(X[y == 1, col], bins=30, alpha=0.6, color="tomato", density=True, label="anomaly")
        ax.set_title(f"{labels[col]}", fontsize=8)
        if col == 0:
            ax.set_ylabel(title, fontsize=7)
        if row == 0 and col == 4:
            ax.legend(fontsize=7)
fig.suptitle("Feature Distributions — Normal (blue) vs Anomaly (red)", fontsize=13)
fig.tight_layout()
fig.savefig(FIGURES / "01_feature_distributions.png", dpi=150, bbox_inches="tight")
plt.close(fig)
print("Saved 01_feature_distributions.png")


## Network dataset: contextual structure

Anomalies have *normal absolute request rates* but occur at unusual times.

In [ ]:
# Reconstruct hour from sin/cos encoding
hour_net = (np.arctan2(X_net[:, 0], X_net[:, 1]) * 24 / (2 * np.pi)) % 24

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for ax, mask, label, color in [
    (axes[0], y_net == 0, "Normal (y=0)", "steelblue"),
    (axes[1], y_net == 1, "Anomaly (y=1)", "tomato"),
]:
    ax.scatter(hour_net[mask], X_net[mask, 2], alpha=0.35, s=8, c=color)
    ax.set_xlabel("Hour of day")
    ax.set_ylabel("Request rate")
    ax.set_title(f"Network — {label}")
    ax.set_xlim(0, 24)
    ax.set_xticks(range(0, 25, 4))

fig.suptitle("Contextual anomaly: high request rate at 1–5 AM looks normal globally", fontsize=11)
fig.tight_layout()
fig.savefig(FIGURES / "01_network_contextual_structure.png", dpi=150, bbox_inches="tight")
plt.close(fig)
print("Saved 01_network_contextual_structure.png")


## Manufacturing dataset: seasonal pattern

Anomalies fall *within the global temperature range* but deviate from the seasonal expectation.

In [ ]:
n_show = min(1500, len(X_mfg))
t_idx = np.arange(n_show)
anom_mask_mfg = y_mfg[:n_show] == 1

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 6), sharex=True)

ax1.plot(t_idx, X_mfg[:n_show, 0], lw=0.7, alpha=0.7, color="steelblue", label="raw temp")
ax1.plot(t_idx, X_mfg[:n_show, 3], lw=1.2, color="orange", label="seasonal expected")
ax1.scatter(np.where(anom_mask_mfg)[0], X_mfg[:n_show, 0][anom_mask_mfg],
            color="red", s=25, zorder=5, label="anomaly")
ax1.set_ylabel("Temperature (°C)")
ax1.legend(fontsize=8, loc="upper right")
ax1.set_title("Raw temperature vs. seasonal expectation")

ax2.plot(t_idx, X_mfg[:n_show, 4], lw=0.7, color="purple", alpha=0.7, label="residual")
ax2.axhline(0, color="k", lw=0.5, ls="--")
ax2.scatter(np.where(anom_mask_mfg)[0], X_mfg[:n_show, 4][anom_mask_mfg],
            color="red", s=25, zorder=5, label="anomaly residual")
ax2.set_ylabel("Residual (°C)")
ax2.set_xlabel("Time step")
ax2.legend(fontsize=8)
ax2.set_title("Residual = raw − seasonal expected  (anomalies are obvious here)")

fig.tight_layout()
fig.savefig(FIGURES / "01_manufacturing_seasonal.png", dpi=150, bbox_inches="tight")
plt.close(fig)
print("Saved 01_manufacturing_seasonal.png")
